In [ ]:
import pandas as pd
from glob import glob
from PIL import Image, ImageDraw
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import random
from ultralytics import YOLO
from tqdm import tqdm
%matplotlib inline

def get_images_labels(path, folder_name):
    """Retrieve sorted lists of label and image file paths for a given folder."""
    labels = glob(f'{path}/train_labels/{folder_name}*')
    labels = sorted(labels)
    imgs = glob(f'{path}/train/{folder_name.split("_")[0]}/{folder_name}*')
    imgs = sorted(imgs)
    return labels, imgs

def generate_random_color():
    """Generate a bright random RGB color."""
    return tuple(random.randint(100, 255) for _ in range(3))

def stack_images_vertically(image_paths):
    """Stack images vertically into a single PIL image, cropping to the minimum width."""
    images = [Image.open(path) for path in image_paths]
    widths, heights = zip(*(i.size for i in images))
    min_width = min(widths)
    total_height = sum(heights)
    stacked_image = Image.new('RGB', (min_width, total_height))
    y_offset = 0
    for im in images:
        if im.width > min_width:
            im = im.crop((0, 0, min_width, im.height))
        stacked_image.paste(im, (0, y_offset))
        y_offset += im.height
    return stacked_image

import os
from PIL import Image

def prepare_stacked_data(image_paths, label_paths):
    """
    Prepare a stacked image and combined mask string from matching image-label pairs.

    Args:
        image_paths (list): List of paths to image files.
        label_paths (list): List of paths to label files.

    Returns:
        tuple: (stacked_image, combined_masks_str)
            - stacked_image: PIL Image object of the stacked images.
            - combined_masks_str: String combining all mask annotations.
    """
    # Create a dictionary mapping label base names to their full paths
    label_map = {os.path.splitext(os.path.basename(lf))[0]: lf for lf in label_paths}
    matching_images = []
    matching_labels = []

    # Match images with corresponding labels
    for image_path in image_paths:
        image_name = os.path.splitext(os.path.basename(image_path))[0]
        if image_name in label_map:
            matching_images.append(image_path)
            matching_labels.append(label_map[image_name])

    # Print feedback for debugging
    # print(f"Found {len(image_paths)} images and {len(label_paths)} labels.")
    # print(f"Matched {len(matching_images)} image-label pairs.")

    # Load and stack the matching images
    images = [Image.open(path) for path in matching_images]
    if len(images) == 0:
        images = [Image.open(path) for path in image_paths]
    widths, heights = zip(*(i.size for i in images))
    min_width = min(widths)
    total_height = sum(heights)

    # Stack images vertically
    stacked_image = Image.new('RGB', (min_width, total_height))
    y_offset = 0
    for im in images:
        im_resized = im.resize((min_width, im.height))
        stacked_image.paste(im_resized, (0, y_offset))
        y_offset += im.height

    # Handle case where no matching pairs are found
    if not matching_images:
        print("No matching image-label pairs found. Just return the image")
        return stacked_image, ""

    # Combine masks with adjusted coordinates
    combined_mask_lines = []
    cumulative_height = 0
    for label_path, im_height, im_width in zip(matching_labels, heights, widths):
        with open(label_path, 'r') as file:
            for line in file:
                parts = line.strip().split()
                if len(parts) < 3:
                    continue
                class_id = parts[0]
                coords = list(map(float, parts[1:]))
                new_coords = []
                for x_norm, y_norm in zip(coords[0::2], coords[1::2]):
                    x_pixel = int(x_norm * im_width)
                    y_pixel = int(y_norm * im_height)
                    x_pixel_cropped = min(x_pixel, min_width - 1)
                    y_pixel_offset = y_pixel + cumulative_height
                    new_x_norm = x_pixel_cropped / min_width
                    new_y_norm = y_pixel_offset / total_height
                    new_coords.extend([new_x_norm, new_y_norm])
                new_line = class_id + " " + " ".join(f"{val:.6f}" for val in new_coords)
                combined_mask_lines.append(new_line)
        cumulative_height += im_height
    combined_masks_str = "\n".join(combined_mask_lines)

    return stacked_image, combined_masks_str

def get_polygons_from_mask_str(mask_str, img_width, img_height):
    polygons = []
    # Split into lines (each line is a polygon)
    lines = mask_str.strip().split('\n') 
    for line in lines:
        # Split line into parts (class ID and coordinates)
        parts = line.strip().split()
        if len(parts) < 3:  # Need at least class ID + one (x, y) pair
            continue
        # Convert coordinates to floats, skipping the class ID
        coords = list(map(float, parts[1:]))
        if len(coords) % 2 != 0:  # Must have even number for (x, y) pairs
            continue
        # Group into (x, y) pairs and convert to pixel coordinates
        points = [(int(x * img_width), int(y * img_height)) 
                  for x, y in zip(coords[0::2], coords[1::2])]
        # Convert to NumPy array and append to list
        polygons.append(np.array(points, dtype=np.int32))
    return polygons
def visualize(stacked_image, polygons):
    # Visualize with polygons
    image = np.array(stacked_image)
    for polygon in polygons:
        if polygon.shape[0]:
            color = generate_random_color()
            cv2.polylines(image, [polygon.astype(int)], isClosed=True, color=color, thickness=1)
    plt.imshow(image)
    plt.axis('off')
    plt.show()

In [ ]:
# Main Execution

# Set paths
PATH = '/raid/ml/root'
train = pd.read_csv(f'{PATH}/Train.csv')
output_dir = f'{PATH}/seg/images'
os.makedirs(output_dir, exist_ok=True)
os.makedirs(output_dir.replace('images', 'labels'), exist_ok=True)

# 1. Organize Data for Training
for f in tqdm(train.FolderName.unique()):
    for side in ['L', 'R']:
        labels, imgs = get_images_labels(PATH, f'{f}_{side}_')
        if imgs and labels:
            stacked_image, mask_str = prepare_stacked_data(imgs, labels)
            output_path = f'{output_dir}/{f}_{side}.png'
            stacked_image.save(output_path)
            with open(output_path.replace('images', 'labels').replace('.png', '.txt'), 'w') as fx:
                fx.write(mask_str)

In [ ]:
image = Image.open(f'{PATH}/seg/images/A2miww5mfx_L.png')
w, h = image.size
with open(f'{PATH}/seg/labels/A2miww5mfx_L.txt') as f:
    mask_str = f.read()
polygons = get_polygons_from_mask_str(mask_str, w, h)
visualize(image, polygons)

In [ ]:
# 2. Create YAML Configuration for YOLO
yaml_content = """
path: /raid/ml/root/seg
train: images
val: images
names:
    0: root
"""
with open(f'./seg/config.yaml', 'w') as f:
    f.write(yaml_content.strip())

# 3. Train the Model on Combined Images
model = YOLO("yolo11l-seg.pt")
model.train(data=f"./seg/config.yaml", epochs=100, imgsz=640)

In [ ]:
# 4. Inference and Visualization Example
# Select example images
folders = train.FolderName.unique()
example_folder = folders[0]
example_side = 'L'
labels, imgs = get_images_labels(PATH, f'{example_folder}_{example_side}_')
stacked_image, _ = prepare_stacked_data(imgs[20:60], labels[20:60])  # Stack first 5 images
stacked_image

In [ ]:
# Load trained model
model = YOLO('./runs/segment/train/weights/best.pt')  # Adjust path if necessary

# Perform inference
results = model(stacked_image)
polygons = results[0].masks.xy
visualize(stacked_image, polygons)